Test LiveImport notebook integration with respect to lazy imports.  

This notebook should be viewed as an extension of `notebook.ipynb`, testing
aspects of notebook integration that cannot be tested when running Python
<3.15.

See `notebook.ipynb` for a description of the `#@` tags.

**NOTE:** Unlike `notebook.ipynb`, this notebook cannot be rerun without
restarting the kernel.  Fresh lazy imports are required so the relevant modules
start out unloaded.

In [ ]:
#
# Note we do not include the usual
#
#   import from setup_imports *
#
# This enables us to lazy import some of those modules later.
#

import liveimport
import sys
from types import LazyImportType, ModuleType
from IPython.display import Markdown

from setup import *

describe_environment()

is_registered = is_registered_fn(globals())

def is_proxy(name:str):
    return isinstance(globals()[name],LazyImportType)

def ok():
    display(Markdown("OK"))

ok()

#
# SCRIPTED_TEST isn't used in this notebook, but integration.py expects it.
#

SCRIPTED_TEST = False

In [ ]:
#
# We set grace to zero for all tests in this notebook so autosync is always
# triggered between cells.  We also make sure there are no implicit lazy
# imports at the start.
#

liveimport.auto_sync(grace=0.0)

__lazy_imports__ = []

ok()

### Keyword Lazy Imports

In [ ]:
#_%%liveimport --clear
lazy import su1a.b                                # type: ignore
lazy import su2a.b as su2ab                       # type: ignore
lazy from su3a.b import x as su3abx, y as su3aby  # type: ignore

assert is_lazy("su1a.b")
assert is_lazy("su2a.b")
assert is_lazy("su3a.b")
assert is_lazy("su3a.b.x")
assert is_lazy("su3a.b.y")

ok()

In [ ]:
#
# The modules referenced by the lazy imports above should have been loaded when
# the statements were registered, but the proxies should have remained
# unresolved.
#

assert is_registered("su1a.b")
assert is_registered("su2a.b",None,"su2ab")
assert is_registered("su3a.b","x","su3abx")
assert is_registered("su3a.b","y","su3aby")

assert not is_lazy("su1a.b")
assert not is_lazy("su2a.b")
assert not is_lazy("su3a.b")
assert not is_lazy("su3a.b.y")

assert is_proxy("su1a")
assert is_proxy("su2ab")
assert is_proxy("su3abx")
assert is_proxy("su3aby")

ok()

In [ ]:
su1ab_tag = get_tag("su1a.b")
su2ab_tag = get_tag("su2a.b")
su3ab_tag = get_tag("su3a.b")
su3aby_tag = get_tag("su3a.b.y")

touch_module("su1a.b")
touch_module("su2a.b")
touch_module("su3a.b")
touch_module("su3a.b.y")

ok()

In [ ]:
expect_tag("su1a.b",next_tag(su1ab_tag))
expect_tag("su2a.b",next_tag(su2ab_tag))
expect_tag("su3a.b",next_tag(su3ab_tag))
expect_tag("su3a.b.y",next_tag(su3aby_tag))

ok()

#@ reload su1a.b
#@ reload su2a.b
#@ reload su3a.b
#@ reload su3a.b.y

`__lazy_modules__` Lazy Imports

In [ ]:
#_%%liveimport --clear

__lazy_modules__ = [ "mod1", "mod2", "pkg.smod2" ]

import mod1                          # type: ignore
from mod2 import mod2_public1        # type: ignore
from pkg.smod2 import smod2_public1  # type: ignore

assert is_lazy("mod1")
assert is_lazy("mod2")
assert is_lazy("mod2.mod2_public1")
assert is_lazy("pkg.smod2")
assert is_lazy("pkg.smod2.smod2_public1")

ok()

In [ ]:
#
# The modules referenced by the implicit lazy imports above should have been
# loaded when the statements were registered, but the proxies should have
# remained unresolved.
#

assert is_registered("mod1")
assert is_registered("mod2","mod2_public1")
assert is_registered("pkg.smod2","smod2_public1")

assert not is_lazy("mod1")
assert not is_lazy("mod2")
assert not is_lazy("pkg.smod2")

assert is_proxy("mod1")
assert is_proxy("mod2_public1")
assert is_proxy("smod2_public1")

ok()

In [ ]:
mod1_tag = get_tag("mod1")
mod2_tag = get_tag("mod2")
smod2_tag = get_tag("pkg.smod2")

touch_module("mod1")
touch_module("mod2")
touch_module("pkg.smod2")

ok()

In [ ]:
expect_tag("mod1",next_tag(mod1_tag))
expect_tag("mod2",next_tag(mod2_tag))
expect_tag("pkg.smod2",next_tag(smod2_tag))

ok()

#@ reload mod1
#@ reload mod2
#@ reload pkg.smod2

### Dependencies

In [ ]:
#
# Make the workspace be the hierarchy under test.
#

import os
liveimport.workspace(root())
ok()

In [ ]:
#_%%liveimport --clear

#
# su4a.b.y lazy imports su5a.b.y, which lazy imports su6a.b.y and su7a.b.y.
#
# NOTE: Observe the import of su4a.b.y is not lazy.
#

import su4a.b.y as su4aby  # type: ignore

ok()

In [ ]:
#
# Dependencies should be not be loaded or tracked yet.  However, su5a.b.y
# should be lazy and watched for reification because su4a.b.y depends on it.
# su[67]a.b.y are not lazy because they are not yet imported in any way.
#

assert is_registered("su4a.b.y",None,"su4aby")

assert "su5a.b.y" not in sys.modules
assert "su6a.b.y" not in sys.modules
assert "su7a.b.y" not in sys.modules
assert is_lazy("su5a.b.y")
assert not is_lazy("su6a.b.y")
assert not is_lazy("su7a.b.y")
assert not is_tracked("su5a.b.y")
assert not is_tracked("su6a.b.y")
assert not is_tracked("su7a.b.y")
assert is_reify_watched("su5a.b.y")
assert not is_reify_watched("su6a.b.y")
assert not is_reify_watched("su7a.b.y")

ok()

In [ ]:
#
# Eagerly import (but do not register) su7a.b.y.
#

import su7a.b.y  # type: ignore

assert "su5a.b.y" not in sys.modules
assert "su6a.b.y" not in sys.modules
assert "su7a.b.y" in sys.modules
assert is_lazy("su5a.b.y")
assert not is_lazy("su6a.b.y")
assert not is_lazy("su7a.b.y")
assert not is_tracked("su5a.b.y")
assert not is_tracked("su6a.b.y")
assert not is_tracked("su7a.b.y")
assert is_reify_watched("su5a.b.y")
assert not is_reify_watched("su6a.b.y")
assert not is_reify_watched("su7a.b.y")

ok()

In [ ]:
#
# Reify the reference to su5a.b.y in su4a.b.y.  This should load su5a.b.y.
#

assert isinstance(su4aby.su5aby, ModuleType)

#
# Even though su5a.b.y is loaded, it will not be tracked yet because there has
# been no reify watch poll. su6a.b.y becomes lazy because the lazy import in
# su5a.b.y executed. (su7a.b.y does not become lazy because it is loaded.)
#

assert "su5a.b.y" in sys.modules
assert "su6a.b.y" not in sys.modules
assert "su7a.b.y" in sys.modules
assert not is_lazy("su5a.b.y")
assert is_lazy("su6a.b.y")
assert not is_lazy("su7a.b.y")
assert not is_tracked("su5a.b.y")
assert not is_tracked("su6a.b.y")
assert not is_tracked("su7a.b.y")
assert is_reify_watched("su5a.b.y")
assert not is_reify_watched("su6a.b.y")
assert not is_reify_watched("su7a.b.y")

ok()

In [ ]:
#
# There should have been a reify watch poll at the end of the preceding cell.
# su5a.b.y should become tracked, causing su7a.b.y to become tracked and
# su6a.b.y watched for reification.
#

assert "su5a.b.y" in sys.modules
assert "su6a.b.y" not in sys.modules
assert "su7a.b.y" in sys.modules
assert not is_lazy("su5a.b.y")
assert is_lazy("su6a.b.y")
assert not is_lazy("su7a.b.y")
assert is_tracked("su5a.b.y")
assert not is_tracked("su6a.b.y")
assert is_tracked("su7a.b.y")
assert not is_reify_watched("su5a.b.y")
assert is_reify_watched("su6a.b.y")
assert not is_reify_watched("su7a.b.y")

ok()

In [ ]:
#
# We touch su6a.b.y's source file, but because it isn't yet loaded, it will not
# reload. Module is su7a.b.y is loaded and tracked, so it will reload, along
# with su5a.b.y and su4a.b.y because of the dependency chain.
#

touch_file("su6a/b/y.py")
touch_module("su7a.b.y")
ok()

In [ ]:
ok()

#@ reload su7a.b.y
#@ reload su5a.b.y
#@ reload su4a.b.y

In [ ]:
#
# After a reify, su6a.b.y reloads when touched.
#

getattr(sys.modules['su5a.b.y'],'su6aby')
ok()

In [ ]:
touch_module("su6a.b.y")
ok()

In [ ]:
ok()

#@ reload su6a.b.y
#@ reload su5a.b.y
#@ reload su4a.b.y

### End of Tests

Make sure to restart before rerunning.